<a href="https://colab.research.google.com/github/anaghahegde326/BIS_LAB/blob/main/BIS_LAB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np

# XOR dataset
X = np.array([[0, 0], [0, 1],
              [1, 0], [1, 1]])

y = np.array([[0], [1], [1], [0]])

# Activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

# Neural network prediction
def predict(p):
    W1 = p[:8].reshape(2, 4)
    b1 = p[8:12]
    W2 = p[12:16].reshape(4, 1)
    b2 = p[16]

    hidden = sigmoid(X @ W1 + b1)
    output = sigmoid(hidden @ W2 + b2)

    return output

# Fitness function
def fitness(p):
    return np.mean((predict(p) - y) ** 2)

# PSO parameters
np.random.seed(42)
particles = 40
iterations = 3000
dimensions = 17

w = 0.7
c1 = 1.5
c2 = 1.5

# Initialize particles
rng = np.random.default_rng(42)
position = rng.uniform(-2, 2, (particles, dimensions))
velocity = rng.uniform(-0.1, 0.1, (particles, dimensions))

pbest = position.copy()
pbest_value = np.array([fitness(p) for p in position])

best_index = np.argmin(pbest_value)
gbest = pbest[best_index].copy()
gbest_value = pbest_value[best_index]

# PSO iterations
for i in range(iterations):

    r1 = rng.random((particles, dimensions))
    r2 = rng.random((particles, dimensions))

    velocity = (
        w * velocity
        + c1 * r1 * (pbest - position)
        + c2 * r2 * (gbest - position)
    )

    position = np.clip(position + velocity, -5, 5)

    for j in range(particles):
        current = fitness(position[j])

        if current < pbest_value[j]:
            pbest[j] = position[j].copy()
            pbest_value[j] = current

    best_index = np.argmin(pbest_value)

    if pbest_value[best_index] < gbest_value:
        gbest = pbest[best_index].copy()
        gbest_value = pbest_value[best_index]

# Final output
print("Best Fitness:", round(gbest_value, 6))
print("\nPredicted Outputs:")

for i in range(4):
    print(X[i], ":", round(predict(gbest)[i, 0], 4))


Best Fitness: 0.006759

Predicted Outputs:
[0 0] : 0.0156
[0 1] : 0.8879
[1 0] : 0.9699
[1 1] : 0.1155


In [2]:

import numpy as np

# XOR dataset
X = np.array([[0, 0], [0, 1],
              [1, 0], [1, 1]])

y = np.array([[0], [1], [1], [0]])

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

# Neural network prediction
def predict(p):
    W1 = p[:8].reshape(2, 4)
    b1 = p[8:12]
    W2 = p[12:16].reshape(4, 1)
    b2 = p[16]

    h = sigmoid(X @ W1 + b1)
    return sigmoid(h @ W2 + b2)

def fitness(p):
    return np.mean((predict(p) - y) ** 2)

# --------------------------------
# 1. PSO TRAINING
# --------------------------------

rng = np.random.default_rng(42)

n = 40
dim = 17
iterations = 3000

position = rng.uniform(-2, 2, (n, dim))
velocity = rng.uniform(-0.1, 0.1, (n, dim))

pbest = position.copy()
pbest_value = np.array([fitness(p) for p in position])

index = np.argmin(pbest_value)
gbest = pbest[index].copy()
gbest_value = pbest_value[index]

for i in range(iterations):

    r1 = rng.random((n, dim))
    r2 = rng.random((n, dim))

    velocity = (
        0.7 * velocity
        + 1.5 * r1 * (pbest - position)
        + 1.5 * r2 * (gbest - position)
    )

    position = np.clip(position + velocity, -5, 5)

    for j in range(n):
        error = fitness(position[j])

        if error < pbest_value[j]:
            pbest[j] = position[j].copy()
            pbest_value[j] = error

    index = np.argmin(pbest_value)

    if pbest_value[index] < gbest_value:
        gbest = pbest[index].copy()
        gbest_value = pbest_value[index]

pso_error = gbest_value
pso_output = predict(gbest)

# --------------------------------
# 2. BACKPROPAGATION TRAINING
# --------------------------------

rng = np.random.default_rng(7)

W1 = rng.normal(0, 0.8, (2, 4))
b1 = np.zeros(4)

W2 = rng.normal(0, 0.8, (4, 1))
b2 = np.zeros(1)

epochs = 20000
lr = 1.0

for epoch in range(epochs):

    h = sigmoid(X @ W1 + b1)
    output = sigmoid(h @ W2 + b2)

    # Output layer error
    d2 = (output - y) * output * (1 - output)

    dW2 = h.T @ d2 / len(X)
    db2 = np.mean(d2, axis=0)

    # Hidden layer error
    d1 = (d2 @ W2.T) * h * (1 - h)

    dW1 = X.T @ d1 / len(X)
    db1 = np.mean(d1, axis=0)

    # Update weights and biases
    W2 -= lr * dW2
    b2 -= lr * db2

    W1 -= lr * dW1
    b1 -= lr * db1

bp_output = sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2)
bp_error = np.mean((bp_output - y) ** 2)

# --------------------------------
# 3. COMPARE RESULTS
# --------------------------------

print("Algorithm       MSE       Accuracy")

pso_accuracy = np.mean(
    (pso_output >= 0.5) == y
) * 100

bp_accuracy = np.mean(
    (bp_output >= 0.5) == y
) * 100

print("PSO          ", round(pso_error, 6),
      "  ", pso_accuracy, "%")

print("Backprop     ", round(bp_error, 6),
      "  ", bp_accuracy, "%")

print("\nPSO Predictions:")
print(np.round(pso_output.flatten(), 4))

print("\nBackprop Predictions:")
print(np.round(bp_output.flatten(), 4))


Algorithm       MSE       Accuracy
PSO           0.006759    100.0 %
Backprop      0.000296    100.0 %

PSO Predictions:
[0.0156 0.8879 0.9699 0.1155]

Backprop Predictions:
[0.0166 0.9832 0.9812 0.0165]
